In [1]:
!pip install selenium
!apt-get update
!apt install chromium-chromedriver
!pip install tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.0/476.0 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.3 MB/s eta 0:00:00
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Ign:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy Release [5,713 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy Release.gpg [793 B]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [3,181 kB]
Get:11 http://archive.ubun

In [2]:
 import csv
import os
import time
from datetime import date, datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException, NoSuchElementException
import shutil
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

# Inicialize o WebDriver
opções = webdriver.ChromeOptions()
opções.add_argument('--headless')
opções.add_argument('--no-sandbox')
opções.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(options=opções)

# Função para navegar até uma página específica
def navegar_para_pagina(url):
    driver.get(url)

# Função para rolar até o final da página
def rolar_ate_final():
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

# Criar pasta de saída se não existir
output_folder = "output"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"\033[92mA pasta {output_folder} foi criada.\033[0m")

# Dicionário de URLs por categoria
urls = {
    "Cozinhas e areas de servico": "https://www.joli.com.br/banheiros-cozinhas-e-areas-de-servico?page=1",
    "Bomboniere" : "https://www.joli.com.br/bomboniere?page=1",
    "Chuveiros e aquecedores" : "https://www.joli.com.br/chuveiros-e-aquecedores?page=1",
    "climatizacao e ventilacao" : "https://www.joli.com.br/climatizacao-e-ventilacao?page=1",
    "decoracao e utilidades domesticas" : "https://www.joli.com.br/decoracao-e-utilidades-domesticas?page=1",
    "Eletrica e iluminacao" : "https://www.joli.com.br/eletrica-e-iluminacao?page=1",
    "Ferramentas" : "https://www.joli.com.br/ferramentas-e-ferragens?page=1",
    "Jardim" : "https://www.joli.com.br/lazer-e-jardim?page=1",
    "Material de construcao" : "https://www.joli.com.br/material-de-construcao?page=1",
    "Material hidraulico" : "https://www.joli.com.br/material-hidraulico?page=1",
    "Pintura e acessorios" : "https://www.joli.com.br/pintura-e-acessorios?page=1",
    "Pisos revistimentos e porcelanatos" : "https://www.joli.com.br/pisos-revestimentos-e-porcelanatos?page=1",
    "Portas e janelas" : "https://www.joli.com.br/portas-e-janelas?page=1",
    "Seguranca e comunicacao" : "https://www.joli.com.br/seguranca-e-comunicacao?page=1"
}

# Encontre os produtos
for categoria, url in urls.items():
    produtos = []
    produtos_carregados = set()  # Usaremos um set para rastrear produtos únicos
    pagina_atual = 1
    tempo_espera_inicial = 2
    incremento_tempo_espera = 2
    tempo_espera_maximo = 10

    print(f"\033[94mIniciando a coleta de produtos da categoria: {categoria}\033[0m")

    while True:
        try:
            # Navegue até a página atual
            url_com_pagina = f"{url.split('?')[0]}?page={pagina_atual}"
            navegar_para_pagina(url_com_pagina)

            # Rolagem e espera para garantir carregamento dos produtos
            while True:
                rolar_ate_final()
                time.sleep(tempo_espera_inicial)
                try:
                    botao_carregar_mais = driver.find_element(By.CSS_SELECTOR, 'footer')
                    if botao_carregar_mais.is_displayed():
                        continue
                except NoSuchElementException:
                    break

            # Aguarde até que os produtos sejam carregados
            produtos_atual = WebDriverWait(driver, tempo_espera_inicial).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.vtex-search-result-3-x-galleryItem'))
            )

            # Adicione os produtos à lista
            novos_produtos = False
            for produto in produtos_atual:
                try:
                    # Coletar informações do produto
                    nome_empresa = "Joli"
                    data_coleta = date.today().strftime("%Y-%m-%d")
                    codigo_produto_element = produto.find_elements(By.CSS_SELECTOR, '.vtex-product-summary-2-x-brandName')
                    codigo_produto = codigo_produto_element[0].get_attribute('textContent').strip() if codigo_produto_element else "0"
                    imagem_element = produto.find_elements(By.CSS_SELECTOR, '.vtex-product-summary-2-x-image')
                    imagem_url = imagem_element[0].get_attribute('src') if imagem_element else ""
                    nome_produto_element = produto.find_elements(By.CSS_SELECTOR, '.vtex-product-summary-2-x-productNameContainer')
                    nome_produto = nome_produto_element[0].get_attribute('textContent').strip() if nome_produto_element else ""
                    moeda_element = produto.find_elements(By.CSS_SELECTOR, '.joli-joli-theme-0-x-price')
                    moeda = moeda_element[0].get_attribute('textContent').strip() if moeda_element else ""
                    url_produto = produto.find_elements(By.CSS_SELECTOR, 'a')[0].get_attribute('href')

                    # Montar a linha de dados do produto
                    linha_produto = [nome_empresa, data_coleta, codigo_produto, imagem_url, nome_produto, moeda, url_produto]

                    # Adicionar à lista de produtos se ainda não estiver na lista
                    if nome_produto not in produtos_carregados:
                        produtos.append(linha_produto)
                        produtos_carregados.add(nome_produto)
                        novos_produtos = True

                except StaleElementReferenceException:
                    # Se o elemento não estiver mais presente, ignore-o
                    continue

            print(f"\033[93mTotal de produtos coletados de {categoria} até agora: {len(produtos)}\033[0m")

            if not novos_produtos:
                print(f"\033[93mNão há mais novos produtos para carregar de {categoria}.\033[0m")
                break

            # Aumentar o tempo de espera para a próxima iteração
            tempo_espera_inicial += incremento_tempo_espera
            if tempo_espera_inicial > tempo_espera_maximo:
                tempo_espera_inicial = tempo_espera_maximo

            # Avance para a próxima página
            pagina_atual += 1

        except TimeoutException:
            # Se não encontrar mais produtos, saia do loop
            print(f"\033[91mNão foi possível encontrar mais produtos em {categoria}.\033[0m")
            break

    # Salvar os produtos em um arquivo CSV com nome da categoria
    nome_arquivo = os.path.join(output_folder, f'produtos_{categoria}.csv')
    with open(nome_arquivo, 'w', newline='', encoding='utf-8') as arquivo_csv:
        escritor_csv = csv.writer(arquivo_csv)
        escritor_csv.writerow(['Nome empresa', 'Data de coleta', 'Código do produto', 'Imagem', 'Nome do produto', 'Moeda', 'URL do produto'])

        for produto in produtos:
            escritor_csv.writerow(produto)

    print(f"\033[92mOs produtos da categoria {categoria} foram salvos em: {nome_arquivo}\033[0m")

# Mover arquivos CSV para o Google Drive
base_destination_dir = '/content/drive/MyDrive/WebScraping/Joli/WebScraping_coleta_Joli'
today = datetime.today().strftime('%Y-%m-%d')
destination_dir = os.path.join(base_destination_dir, today)

if not os.path.exists(destination_dir):
    os.makedirs(destination_dir, exist_ok=True)

source_dir = output_folder

for file_name in os.listdir(source_dir):
    full_file_name = os.path.join(source_dir, file_name)
    if os.path.isfile(full_file_name):
        shutil.copy(full_file_name, destination_dir)

print(f"\033[92mArquivos CSV movidos para: {destination_dir}\033[0m")

# Feche o driver
driver.quit()

Mounted at /content/drive
A pasta output foi criada.
Iniciando a coleta de produtos da categoria: Cozinhas e areas de servico


KeyboardInterrupt: 